In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import spearmanr, pearsonr
import matplotlib.dates as mdates

Hypothesis H-A: Subjective and objective sleep quality (TST, SE, SOL, WASO) show substantial correlation in within-subject settings.

Single subject dataset (similar to QSci/QS-Carsten)

In [ ]:
strength = 20  # a slider [0, 100] that determines the amount of randomness in data;
               # 0: more random, less correlation; 100: less random: more correlation
rng = np.random.default_rng(9)  # to keep things reproducible

In [ ]:
start_date = "2000-01-01"
end_date   = "2003-12-30"
dates = pd.date_range(start=start_date, end=end_date, freq="D").strftime('%Y-%m-%d')
n = len(dates)

# Subjective Likert-scale sleep quality (1..5)
dataset_n1 = pd.DataFrame({
    "date": dates,
    "Sub_id": 1,
    "Subj_likert": rng.integers(1, 6, size=n)
})

# Map Likert levels to slider bounds and sample a continuous subjective slider score [0,100]
bounds = {1: (0.0, 19.0), 2: (20.0, 39.0), 3: (40.0, 59.0), 4: (60.0, 79.99), 5: (80.0, 100.0)}
low  = np.vectorize(lambda x: bounds[int(x)][0])(dataset_n1["Subj_likert"].values)
high = np.vectorize(lambda x: bounds[int(x)][1])(dataset_n1["Subj_likert"].values)
dataset_n1["Subj_slide"] = np.round(rng.uniform(low, high), 2)

print("Mock QSci data:\n")
print(dataset_n1.iloc[:20, :].to_string(index=False))

Mock QSci data:

      date  Sub_id  Subj_likert  Subj_slide
2000-01-01       1            3       56.58
2000-01-02       1            5       91.30
2000-01-03       1            5       92.44
2000-01-04       1            2       34.23
2000-01-05       1            1        5.76
2000-01-06       1            4       69.49
2000-01-07       1            4       75.37
2000-01-08       1            4       60.92
2000-01-09       1            4       60.62
2000-01-10       1            4       65.96
2000-01-11       1            5       87.68
2000-01-12       1            5       82.54
2000-01-13       1            5       81.15
2000-01-14       1            5       85.58
2000-01-15       1            4       73.09
2000-01-16       1            5       83.70
2000-01-17       1            1        7.70
2000-01-18       1            1       12.64
2000-01-19       1            4       70.63
2000-01-20       1            3       45.75


In [ ]:
# Helper to convert Likert to a quality score in [0,1]
def quality_from_likert(likert):
    return (likert - 1) / 4.0

# Generate objective sleep metrics with tunable correlation to subjective quality
def synthesize_metric(q, low_val, high_val, strength_0_100=0, positive=True, jitter=0.0):
    s = np.clip(strength_0_100 / 100.0, 0.0, 1.0)
    base_noise = rng.random(len(q))
    trend = q if positive else (1.0 - q)
    z = (1.0 - s) * base_noise + s * trend
    if jitter > 0:
        z = np.clip(z + rng.normal(0, jitter, size=len(z)), 0, 1)
    out = low_val + z * (high_val - low_val)
    return out

In [ ]:
# Generating random values for objevtive metrics
target_q = quality_from_likert(dataset_n1["Subj_likert"].values)
# TST: minutes, higher is better. [~300–600]
dataset_n1["TST"] = synthesize_metric(target_q, low_val=300, high_val=600,
                                      strength_0_100=strength, positive=True, jitter=0.01).round().astype(int)

# SE: proportion, higher is better. [~0.60–0.98].
dataset_n1["SE"] = np.round(
    synthesize_metric(target_q, low_val=0.50, high_val=1,
                      strength_0_100=strength, positive=True, jitter=0.01),3)

# SOL: minutes, lower is better. [~30–100].
dataset_n1["SOL"] = synthesize_metric(target_q, low_val=30, high_val=100,
                                      strength_0_100=strength, positive=False, jitter=0.01).round().astype(int)

# WASO: minutes, lower is better.[~10–100].
dataset_n1["WASO"] = synthesize_metric(target_q, low_val=10, high_val=100,
                                       strength_0_100=strength, positive=False, jitter=0.01).round().astype(int)
print("Mock QSci data:\n")
print(dataset_n1.iloc[0:10, :].to_string(index=False))

Mock QSci data:

      date  Sub_id  Subj_likert  Subj_slide  TST    SE  SOL  WASO
2000-01-01       1            3       56.58  533 0.914   40    41
2000-01-02       1            5       91.30  570 0.823   51    65
2000-01-03       1            5       92.44  535 0.952   84    11
2000-01-04       1            2       34.23  482 0.693   76    93
2000-01-05       1            1        5.76  406 0.661   47    34
2000-01-06       1            4       69.49  386 0.748   77    36
2000-01-07       1            4       75.37  540 0.899   47    82
2000-01-08       1            4       60.92  429 0.730   37    70
2000-01-09       1            4       60.62  500 0.703   86    40
2000-01-10       1            4       65.96  359 0.741   42    27


In [ ]:
# Fisher z-transform CI for Pearson's r (assumes independence; OK for weekly; daily CI additionally via block bootstrap)
def fisher_ci(r, n, alpha=0.05):
    if n < 4 or np.isclose(1-abs(r), 0):
        return (np.nan, np.nan)
    z = np.arctanh(np.clip(r, -0.999999, 0.999999))
    se = 1 / np.sqrt(n - 3)
    z_lo = z - 1.96 * se
    z_hi = z + 1.96 * se
    return (np.tanh(z_lo), np.tanh(z_hi))

# Simple moving block bootstrap CI for correlation (serial-aware for daily)
rng_boot = np.random.default_rng(42)
def block_boot_ci(x, y, method="spearman", block=7, reps=2000, alpha=0.05):
    x = np.asarray(x, float)
    y = np.asarray(y, float)
    ok = ~np.isnan(x) & ~np.isnan(y)
    x, y = x[ok], y[ok]
    n = len(x)
    if n == 0:
        return (np.nan, np.nan, np.nan)
    if n < max(3, 2*block):
        block = 1

    def draw_idx():
        out = []
        k = 0
        while k < n:
            start = rng_boot.integers(0, n - block + 1)
            out.append(np.arange(start, start + block))
            k += block
        return np.concatenate(out)[:n]

    if method == "spearman":
        corr = lambda a, b: spearmanr(a, b).correlation
    else:
        corr = lambda a, b: pearsonr(a, b)[0]

    stat_hat = corr(x, y)
    boot = np.empty(reps)
    for b in range(reps):
        idx = draw_idx()
        boot[b] = corr(x[idx], y[idx])

    lo, hi = np.percentile(boot, [100*alpha/2, 100*(1-alpha/2)])
    return float(stat_hat), float(lo), float(hi)

In [ ]:
# del dataset_n1

In [ ]:
df = dataset_n1.copy()
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date").reset_index(drop=True)
metrics = ["TST", "SE", "SOL", "WASO"]
targets = ["Subj_slide", "Subj_likert"]

In [ ]:
# Make column names tidy and discover what actually exists
df.columns = df.columns.str.strip()

# Define expected names, then keep only those that exist in df
metrics_all = ["TST", "SE", "SOL", "WASO"]
targets_all = ["Subj_slide", "Subj_scale", "Subj_likert"]  # include both slider spellings

metrics  = [c for c in metrics_all if c in df.columns]
targets  = [c for c in targets_all if c in df.columns]

if not metrics or not targets:
    raise ValueError(f"No usable metrics/targets found. df has: {list(df.columns)}")

In [ ]:
# Daily correlations (one row per metric × target):

daily_rows = []
for metric in metrics:
    for target in targets:
        # Spearman
        rho_hat, rho_lo, rho_hi = block_boot_ci(df[metric], df[target], method="spearman", block=7, reps=2000)
        # Pearson
        r_hat, r_lo, r_hi = block_boot_ci(df[metric], df[target], method="pearson", block=7, reps=2000)
        daily_rows.append({
            "metric": metric,
            "target": target,
            "n_obs": int(df[[metric, target]].dropna().shape[0]),
            "spearman_rho": rho_hat,
            "spearman_ci_lo": rho_lo,
            "spearman_ci_hi": rho_hi,
            "pearson_r": r_hat,
            "pearson_ci_lo": r_lo,
            "pearson_ci_hi": r_hi
        })

daily_results = pd.DataFrame(daily_rows)
daily_results[["spearman_rho","spearman_ci_lo","spearman_ci_hi",
               "pearson_r","pearson_ci_lo","pearson_ci_hi"]] = daily_results[
    ["spearman_rho","spearman_ci_lo","spearman_ci_hi","pearson_r","pearson_ci_lo","pearson_ci_hi"]].round(4)

print("\nDaily correlations with 95% CI (block bootstrap):\n")
print(daily_results.to_string(index=False))


Daily correlations with 95% CI (block bootstrap):

metric      target  n_obs  spearman_rho  spearman_ci_lo  spearman_ci_hi  pearson_r  pearson_ci_lo  pearson_ci_hi
   TST  Subj_slide   1460        0.2721          0.2205          0.3220     0.2853         0.2369         0.3311
   TST Subj_likert   1460        0.2731          0.2228          0.3199     0.2859         0.2338         0.3348
    SE  Subj_slide   1460        0.2459          0.1954          0.2893     0.2576         0.2126         0.3004
    SE Subj_likert   1460        0.2450          0.1994          0.2900     0.2569         0.2108         0.3022
   SOL  Subj_slide   1460       -0.2589         -0.3088         -0.2084    -0.2787        -0.3297        -0.2322
   SOL Subj_likert   1460       -0.2732         -0.3201         -0.2257    -0.2921        -0.3401        -0.2464
  WASO  Subj_slide   1460       -0.2850         -0.3299         -0.2389    -0.2967        -0.3424        -0.2515
  WASO Subj_likert   1460       -0.2914     

In [ ]:
# Weekly aggregation
weekly = (
    df.set_index("date")
      .resample("W-MON")  # weeks ending on Monday (same as original)
      .agg({
          "TST": "mean",
          "SE": "mean",
          "SOL": "mean",
          "WASO": "mean",
          "Subj_slide": "mean",
          "Subj_likert": "median"  # median for Likert scale
      }))
weekly = weekly.dropna(how="all").reset_index()
print(weekly.iloc[0:10, :].to_string(index=False))

      date        TST       SE       SOL      WASO  Subj_slide  Subj_likert
2000-01-03 546.000000 0.896333 58.333333 39.000000   80.106667          5.0
2000-01-10 443.142857 0.739286 58.857143 54.571429   53.192857          4.0
2000-01-17 434.142857 0.769286 58.285714 31.571429   71.634286          5.0
2000-01-24 461.428571 0.826571 65.285714 70.857143   51.512857          3.0
2000-01-31 411.714286 0.721286 77.714286 52.285714   45.501429          2.0
2000-02-07 453.000000 0.720714 65.857143 43.428571   69.095714          4.0
2000-02-14 485.571429 0.647857 60.571429 44.571429   63.871429          4.0
2000-02-21 487.285714 0.832429 58.714286 47.000000   71.924286          5.0
2000-02-28 471.857143 0.763429 66.428571 65.714286   63.754286          4.0
2000-03-06 399.285714 0.727857 75.142857 38.142857   46.480000          3.0


In [ ]:
# Weekly correlations

weekly_rows = []
for metric in metrics:
    for target in targets:
        # Spearman
        rho_hat_w, rho_lo_w, rho_hi_w = block_boot_ci(weekly[metric], weekly[target],
                                                      method="spearman", block=1, reps=1000)
        # Pearson
        mask = weekly[[metric, target]].dropna()
        r_w = pearsonr(mask[metric], mask[target])[0] if len(mask) >= 3 else np.nan
        lo_w, hi_w = fisher_ci(r_w, len(mask)) if len(mask) >= 4 else (np.nan, np.nan)
        weekly_rows.append({
            "metric": metric,
            "target": target,
            "n_weeks": int(mask.shape[0]),
            "spearman_rho": rho_hat_w,
            "spearman_ci_lo": rho_lo_w,
            "spearman_ci_hi": rho_hi_w,
            "pearson_r": r_w,
            "pearson_ci_lo": lo_w,
            "pearson_ci_hi": hi_w
        })

weekly_results = pd.DataFrame(weekly_rows)
weekly_results[["spearman_rho","spearman_ci_lo","spearman_ci_hi",
                "pearson_r","pearson_ci_lo","pearson_ci_hi"]] = weekly_results[
    ["spearman_rho","spearman_ci_lo","spearman_ci_hi","pearson_r","pearson_ci_lo","pearson_ci_hi"]].round(4)

print("\nWeekly correlations with 95% CI (Spearman via bootstrap; Pearson via Fisher z):\n")
print(weekly_results.to_string(index=False))



Weekly correlations with 95% CI (Spearman via bootstrap; Pearson via Fisher z):

metric      target  n_weeks  spearman_rho  spearman_ci_lo  spearman_ci_hi  pearson_r  pearson_ci_lo  pearson_ci_hi
   TST  Subj_slide      210        0.2560          0.1200          0.3673     0.3020         0.1737         0.4202
   TST Subj_likert      210        0.2971          0.1671          0.4141     0.3161         0.1888         0.4330
    SE  Subj_slide      210        0.3129          0.1872          0.4316     0.3385         0.2129         0.4532
    SE Subj_likert      210        0.2354          0.1104          0.3628     0.2541         0.1229         0.3765
   SOL  Subj_slide      210       -0.2328         -0.3469         -0.1101    -0.2355        -0.3594        -0.1034
   SOL Subj_likert      210       -0.1933         -0.3133         -0.0582    -0.2061        -0.3322        -0.0728
  WASO  Subj_slide      210       -0.2696         -0.4011         -0.1253    -0.3172        -0.4340        -0.190

Explaination of the results:

eg, Weekly Spearman correlation between TST and subjective score (slider) was 0.953 (95% CI 0.932–0.9654); |ρ|≥0.20; accpeted

###################################

In [ ]:
# Completely random data, no seed

start_date = "2000-01-01"
end_date = "2003-12-31"  # interpreted from user's "2003-12-32"
rng = np.random.default_rng(42)

# Create date range and random data

n = len(dates)

qsci = pd.DataFrame({
    "date": dates,
    "Sub_id": 1,
    "TST": rng.integers(300, 601, size=n),
    "SE": np.round(rng.random(n), 3),
    "SOL": rng.integers(10, 101, size=n),
    "WASO": rng.integers(10, 101, size=n),
    "Subj": rng.integers(1, 6, size=n)
})
df = qsci